# CaseHOLD logprob scoring — Saul-7B-Instruct + (bare | BM25 | SONAR)

**Purpose:** companion to the local letter-emit notebooks. Scores each option
by conditional log-likelihood under Saul instead of asking the model to emit a
letter, which bypasses the A-bias documented in the local notebooks' READMEs.

**Why no fairseq2 here:** SONAR embeddings are deterministic. They are
precomputed once on the project owner's local machine and shipped as `.npy`
files in the repo. Colab downloads them at startup; no fairseq2,
sonar-space, transformers, or torchvision install needed.

**Runtime:** select **T4 GPU** (Runtime → Change runtime type → T4 GPU).
Free tier is enough. End-to-end runtime at the locked
`N_TEST=50, CORPUS_SIZE=2000`: ~15 min after model download.

**Output:** `logprob_results.json` (auto-downloads to your machine). Drop it
back into the local `notebooks/legal-rag/results/` directory; the team
notebooks pick it up as the authoritative eval baseline.

---

## 1. Setup (lightweight)

In [ ]:
# llama-cpp-python with prebuilt CUDA wheel + minimal Python deps
\!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python
\!pip install -q 'datasets<4' rank-bm25 nltk huggingface_hub
print('pip done')

In [ ]:
import os, json, time, sys, urllib.request
import numpy as np
from pathlib import Path
from collections import Counter
print('python', sys.version.split()[0])
import torch
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Config

In [ ]:
SEED         = 42
N_TEST       = 50
K_RETRIEVE   = 5
CORPUS_SIZE  = 2000
N_CTX        = 8192
LETTERS      = 'ABCDE'

HF_REPO  = 'mradermacher/Saul-7B-Instruct-v1-GGUF'
HF_FILE  = 'Saul-7B-Instruct-v1.Q5_K_M.gguf'

# Precomputed SONAR embedding files in the repo
GH_RAW = 'https://raw.githubusercontent.com/nicholasdhaliwal/capstone-research/main/notebooks/legal-rag/data_for_colab'
DATA_DIR = Path('/content/data_for_colab')
DATA_DIR.mkdir(exist_ok=True)

OUT_PATH = '/content/logprob_results.json'
np.random.seed(SEED)

## 3. Download precomputed SONAR embeddings

In [ ]:
files = [
    f'sonar_corpus_{CORPUS_SIZE}.npy',
    f'sonar_corpus_{CORPUS_SIZE}_meta.json',
    f'sonar_test_queries_{N_TEST}.npy',
    'manifest.json',
]
for f in files:
    dst = DATA_DIR / f
    if dst.exists():
        print(f'  cached: {f}')
        continue
    url = f'{GH_RAW}/{f}'
    t0 = time.time()
    urllib.request.urlretrieve(url, dst)
    print(f'  downloaded {f} ({dst.stat().st_size//1024} KB, {time.time()-t0:.1f}s)')

manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
print('\nmanifest:', json.dumps(manifest, indent=2))

## 4. Download Saul Q5_K_M GGUF (~5 GB)

In [ ]:
from huggingface_hub import hf_hub_download
t0 = time.time()
gguf_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
print(f'Saul GGUF at {gguf_path}  ({time.time()-t0:.0f}s)')

## 5. Load CaseHOLD + build BM25

In [ ]:
# Load CaseHOLD directly from HF CSV files (avoids the deprecated loader script).
# Works on any `datasets` version.
from datasets import load_dataset
CASEHOLD_BASE = 'https://huggingface.co/datasets/casehold/casehold/resolve/main/data/all'
ds = load_dataset('csv', data_files={
    'train': f'{CASEHOLD_BASE}/train.csv',
    'test':  f'{CASEHOLD_BASE}/test.csv',
})

# CSV columns are positional. Header is 'Unnamed: 0,0,1,2,...,11'.
# Per the original casehold.py: col 0=example_id, col 1=citing_prompt,
# cols 2..6=holding_0..4, col 12=label.
def _rename(ex):
    return {
        'example_id':    int(ex['Unnamed: 0']),
        'citing_prompt': ex['0'],
        'holding_0':     ex['1'],
        'holding_1':     ex['2'],
        'holding_2':     ex['3'],
        'holding_3':     ex['4'],
        'holding_4':     ex['5'],
        'label':         int(ex['11']),
    }
ds = ds.map(_rename, remove_columns=ds['train'].column_names)

train = ds['train'].select(range(CORPUS_SIZE))
test = ds['test'].select(range(N_TEST))
corpus_texts = [r['citing_prompt'] for r in train]
print(f'corpus: {len(corpus_texts)}  test: {len(test)}')

In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([t.lower().split() for t in corpus_texts])

def retrieve_bm25(q, k=K_RETRIEVE):
    s = bm25.get_scores(q.lower().split())
    return [corpus_texts[i] for i in np.argsort(s)[-k:][::-1]]
print('bm25 ready')

## 6. Build SONAR retriever from precomputed embeddings

`sentence_to_parent[i]` maps sentence-index `i` to its parent excerpt-index.
Query embeddings are precomputed too (the queries are fixed by `N_TEST`),
so retrieval is just a normalized matmul + argsort. No SONAR install needed.

In [ ]:
sent_vec = np.load(DATA_DIR / f'sonar_corpus_{CORPUS_SIZE}.npy')
sent_vec = sent_vec / np.linalg.norm(sent_vec, axis=1, keepdims=True)
meta = json.loads((DATA_DIR / f'sonar_corpus_{CORPUS_SIZE}_meta.json').read_text())
sentence_to_parent = meta['sentence_to_parent']
assert len(sentence_to_parent) == sent_vec.shape[0], 'meta mismatch'
print(f'corpus embeddings: {sent_vec.shape}, parent map: {len(sentence_to_parent)}')

query_vec = np.load(DATA_DIR / f'sonar_test_queries_{N_TEST}.npy')
query_vec = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)
assert query_vec.shape[0] == N_TEST, f'expected {N_TEST} queries, got {query_vec.shape[0]}'
print(f'query embeddings: {query_vec.shape}')

def retrieve_sonar(qi, k=K_RETRIEVE):
    """qi is the test-question index (0..N_TEST-1)."""
    sims = (query_vec[qi:qi+1] @ sent_vec.T)[0]
    order = np.argsort(-sims)
    seen, result = set(), []
    for i in order:
        parent = sentence_to_parent[int(i)]
        if parent not in seen:
            seen.add(parent); result.append(corpus_texts[parent])
            if len(result) == k: break
    return result

print('sonar retriever ready (precomputed embeddings, no fairseq2)')

## 7. Load Saul into llama-cpp-python on GPU

In [ ]:
from llama_cpp import Llama
t0 = time.time()
llm = Llama(
    model_path=gguf_path, n_ctx=N_CTX, n_gpu_layers=-1,
    logits_all=False, verbose=False, seed=SEED,
)
print(f'Saul loaded in {time.time()-t0:.0f}s')

## 8. Logprob scoring

For each option, compute log p(option | prompt) = Σ log p(token_i | prompt, token_{<i}).
Returns both raw `sum_logp` and length-normalized `mean_logp` (lm-evaluation-harness reports both as `acc` and `acc_norm`).

In [ ]:
def build_prompt(excerpt, retrieved):
    ctx = ''
    if retrieved:
        ctx = 'Similar cases:\n' + '\n\n'.join(f'{i+1}. {d}' for i, d in enumerate(retrieved)) + '\n\n'
    return f'{ctx}Excerpt: {excerpt}\n\nThe correct holding is: '

def score_option(prompt, option):
    full = prompt + option
    full_tokens = llm.tokenize(full.encode('utf-8'), add_bos=True)
    prompt_tokens = llm.tokenize(prompt.encode('utf-8'), add_bos=True)
    option_tokens = full_tokens[len(prompt_tokens):]
    if not option_tokens:
        return -1e9, 0
    llm.reset()
    llm.eval(prompt_tokens)
    sum_logp = 0.0
    for tok in option_tokens:
        logits = np.array(llm.eval_logits[-1], dtype=np.float64)
        mx = logits.max()
        log_softmax = logits - (mx + np.log(np.exp(logits - mx).sum()))
        sum_logp += float(log_softmax[tok])
        llm.eval([tok])
    return sum_logp, len(option_tokens)

## 9. Eval loop

In [ ]:
def get_retrieved(rname, qi, q_text):
    if rname == 'bare':  return None
    if rname == 'bm25':  return retrieve_bm25(q_text)
    if rname == 'sonar': return retrieve_sonar(qi)

records = []
for qi, row in enumerate(test):
    q = row['citing_prompt']
    gold = int(row['label'])
    options = [row[f'holding_{i}'] for i in range(5)]
    rec = {'question_id': qi, 'gold': LETTERS[gold], 'results': []}
    for rname in ['bare', 'bm25', 'sonar']:
        retrieved = get_retrieved(rname, qi, q)
        prompt = build_prompt(q, retrieved)
        t0 = time.time()
        scores = []
        for j, opt in enumerate(options):
            sl, n = score_option(prompt, opt)
            scores.append({'letter': LETTERS[j], 'sum_logp': sl, 'n_tokens': n, 'mean_logp': sl / max(n, 1)})
        pred_raw  = LETTERS[int(np.argmax([s['sum_logp']  for s in scores]))]
        pred_norm = LETTERS[int(np.argmax([s['mean_logp'] for s in scores]))]
        rec['results'].append({
            'retriever': rname,
            'pred_raw': pred_raw, 'pred_norm': pred_norm,
            'correct_raw':  pred_raw  == LETTERS[gold],
            'correct_norm': pred_norm == LETTERS[gold],
            'scores': scores,
            'latency_s': round(time.time()-t0, 2),
        })
    records.append(rec)
    if (qi + 1) % 5 == 0: print(f'  {qi+1}/{N_TEST}')
print('eval done')

## 10. Accuracy summary

In [ ]:
print('=== LOGPROB SUMMARY (raw + length-normalized) ===')
for rname in ['bare', 'bm25', 'sonar']:
    rs = [r for rec in records for r in rec['results'] if r['retriever'] == rname]
    acc_raw  = sum(r['correct_raw']  for r in rs) / len(rs)
    acc_norm = sum(r['correct_norm'] for r in rs) / len(rs)
    lc_raw   = Counter(r['pred_raw']  for r in rs)
    lc_norm  = Counter(r['pred_norm'] for r in rs)
    lat_mean = np.mean([r['latency_s'] for r in rs])
    print(f'\n{rname}: acc_raw={acc_raw:.3f}  acc_norm={acc_norm:.3f}  lat_mean={lat_mean:.1f}s')
    print(f'  pred_raw dist:  {dict(lc_raw)}')
    print(f'  pred_norm dist: {dict(lc_norm)}')

## 11. Save + download `logprob_results.json`

Drop this file back into the local repo at
`notebooks/legal-rag/results/logprob_results.json`. The team notebooks load
it as the authoritative eval baseline.

In [ ]:
payload = {
    'config': {
        'seed': SEED, 'n_test': N_TEST, 'k_retrieve': K_RETRIEVE,
        'corpus_size': CORPUS_SIZE, 'n_ctx': N_CTX,
        'model': HF_FILE,
        'sonar_encoder': 'text_sonar_basic_encoder (precomputed locally)',
    },
    'records': records,
}
Path(OUT_PATH).write_text(json.dumps(payload, indent=2))
print(f'wrote {OUT_PATH}  ({Path(OUT_PATH).stat().st_size // 1024} KB)')

In [ ]:
from google.colab import files
files.download(OUT_PATH)